<a href="https://colab.research.google.com/github/IsmaeelDhikr/IsmaeelDhikr/blob/main/DTA_a6_w4_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://cognitiveclass.ai/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkML0201ENSkillsNetwork953-2023-01-01"><img src="https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/Logos/organization_logo/organization_logo.png" width="400"> </a>


Welcome to the final project of  “Apache Spark for Scalable Machine Learning on BigData”. In this assignment you’ll analyze a real-world dataset and apply machine learning on it using Apache Spark.

In order to pass, you need to implement some code (as described in the instruction section on Coursera) and finally answer a quiz on the Coursera platform.


This notebook is designed to run in a IBM Watson Studio default runtime (NOT the Watson Studio Apache Spark Runtime as the default runtime with 1 vCPU is free of charge). Therefore, we install Apache Spark in local mode for test purposes only. Please don't use it in production.

In case you are facing issues, please read the following two documents first:

https://github.com/IBM/skillsnetwork/wiki/Environment-Setup

https://github.com/IBM/skillsnetwork/wiki/FAQ

Then, please feel free to ask:

https://coursera.org/learn/machine-learning-big-data-apache-spark/discussions/all

Please make sure to follow the guidelines before asking a question:

https://github.com/IBM/skillsnetwork/wiki/FAQ#im-feeling-lost-and-confused-please-help-me


If running outside Watson Studio, this should work as well. In case you are running in an Apache Spark context outside Watson Studio, please remove the Apache Spark setup in the first notebook cells.


In [16]:
from IPython.display import Markdown, display
def printmd(string):
    display(Markdown('# <span style="color:red">'+string+'</span>'))


if ('sc' in locals() or 'sc' in globals()):
    printmd('<<<<<!!!!! It seems that you are running in a IBM Watson Studio Apache Spark Notebook. Please run it in an IBM Watson Studio Default Runtime (without Apache Spark) !!!!!>>>>>')


# <span style="color:red"><<<<<!!!!! It seems that you are running in a IBM Watson Studio Apache Spark Notebook. Please run it in an IBM Watson Studio Default Runtime (without Apache Spark) !!!!!>>>>></span>

In [17]:
!pip install pyspark==2.4.5

  Using cached pyspark-2.4.5.tar.gz (217.8 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> pyspark

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [18]:
try:
    from pyspark import SparkContext, SparkConf
    from pyspark.sql import SparkSession
except ImportError as e:
    printmd('<<<<<!!!!! Please restart your kernel after installing Apache Spark !!!!!>>>>>')

In [19]:
sc = SparkContext.getOrCreate(SparkConf().setMaster("local[*]"))

spark = SparkSession \
    .builder \
    .getOrCreate()

Let’s start by downloading the dataset and creating a dataframe. This dataset can be found on DAX, the IBM Data Asset Exchange and can be downloaded for free.

https://developer.ibm.com/exchanges/data/all/jfk-weather-data/


In [25]:
# delete files from previous runs
!rm -rf noaa-weather-data-jfk-airport*

df = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv("/content/LCD_USW00094789_2020.csv")


)

# register a corresponding query table
df.createOrReplaceTempView('df')

The dataset contains some null values, therefore schema inference didn’t work properly for all columns, in addition, a column contained trailing characters, so we need to clean up the data set first. This is a normal task in any data science project since your data is never clean, don’t worry if you don’t understand all code, you won’t be asked about it.


In [27]:
import random
random.seed(42)

from pyspark.sql.functions import translate, col

# Map the CSV columns to the uppercase column names used in the notebook
df_cleaned = df \
    .withColumn("HOURLYWindSpeed", col("HourlyWindSpeed").cast('double')) \
    .withColumn("HOURLYWindDirection", col("HourlyWindDirection").cast('double')) \
    .withColumn("HOURLYStationPressure", translate(col("HourlyStationPressure"), "s,", "")) \
    .withColumn("HOURLYPrecip", translate(col("HourlyPrecipitation"), "s,", "")) \
    .withColumn("HOURLYRelativeHumidity", translate(col("HourlyRelativeHumidity"), "*", "")) \
    .withColumn("HOURLYDRYBULBTEMPC", translate(col("HourlyDryBulbTemperature"), "*", "")) \

df_cleaned = df_cleaned \
    .withColumn("HOURLYStationPressure", df_cleaned.HOURLYStationPressure.cast('double')) \
    .withColumn("HOURLYPrecip", df_cleaned.HOURLYPrecip.cast('double')) \
    .withColumn("HOURLYRelativeHumidity", df_cleaned.HOURLYRelativeHumidity.cast('double')) \
    .withColumn("HOURLYDRYBULBTEMPC", df_cleaned.HOURLYDRYBULBTEMPC.cast('double')) \

df_filtered = df_cleaned.filter("""
    HOURLYWindSpeed <> 0
    and HOURLYWindSpeed IS NOT NULL
    and HOURLYWindDirection IS NOT NULL
    and HOURLYStationPressure IS NOT NULL
    and HourlyPressureTendency IS NOT NULL
    and HOURLYPrecip IS NOT NULL
    and HOURLYRelativeHumidity IS NOT NULL
    and HOURLYDRYBULBTEMPC IS NOT NULL
""")

We want to predict the value of one column based of some others. It is sometimes helpful to print a correlation matrix.


In [31]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import when, col, translate

# Redefine the pipeline cleaning step securely
# 1. Handle 'VRB' in wind direction by turning it to Null (or 0.0) before casting
# 2. Handle 'T' in precipitation before casting
df_cleaned_safe = df \
    .withColumn("HOURLYWindSpeed", col("HourlyWindSpeed").cast('double')) \
    .withColumn("HOURLYWindDirectionRaw", col("HourlyWindDirection")) \
    .withColumn("HOURLYWindDirection", when(col("HOURLYWindDirectionRaw") == "VRB", None).otherwise(col("HOURLYWindDirectionRaw")).cast('double')) \
    .withColumn("HOURLYStationPressure", translate(col("HourlyStationPressure"), "s,", "").cast('double')) \
    .withColumn("HOURLYPrecipRaw", translate(col("HourlyPrecipitation"), "s,", "")) \
    .withColumn("HOURLYPrecip", when(col("HOURLYPrecipRaw") == "T", "0.0").otherwise(col("HOURLYPrecipRaw")).cast('double')) \
    .withColumn("HOURLYRelativeHumidity", translate(col("HourlyRelativeHumidity"), "*", "").cast('double')) \
    .withColumn("HOURLYDRYBULBTEMPC", translate(col("HourlyDryBulbTemperature"), "*", "").cast('double'))

df_filtered_safe = df_cleaned_safe.filter("""
    HOURLYWindSpeed <> 0
    and HOURLYWindSpeed IS NOT NULL
    and HOURLYWindDirection IS NOT NULL
    and HOURLYStationPressure IS NOT NULL
    and HourlyPressureTendency IS NOT NULL
    and HOURLYPrecip IS NOT NULL
    and HOURLYRelativeHumidity IS NOT NULL
    and HOURLYDRYBULBTEMPC IS NOT NULL
""")

vectorAssembler = VectorAssembler(inputCols=["HOURLYWindSpeed", "HOURLYWindDirection", "HOURLYStationPressure"],
                                  outputCol="features")
df_pipeline = vectorAssembler.transform(df_filtered_safe)

from pyspark.ml.stat import Correlation
Correlation.corr(df_pipeline, "features").head()[0].toArray()

array([[ 1.        ,  0.22521762, -0.25786296],
       [ 0.22521762,  1.        , -0.13248625],
       [-0.25786296, -0.13248625,  1.        ]])

As we can see, HOURLYWindSpeed and HOURLYWindDirection correlate with 0.25478863 whereas HOURLYWindSpeed  and HOURLYStationPressure correlate with -0.26171147, this is a good sign if we want to predict HOURLYWindSpeed from HOURLYWindDirection and HOURLYStationPressure. Note that the numbers can change over time as we are always working with the latest data.
Since this is supervised learning, let’s split our data into train (80%) and test (20%) set.


In [32]:
splits = df_filtered.randomSplit([0.8, 0.2])
df_train = splits[0]
df_test = splits[1]

Again, we can re-use our feature engineering pipeline


In [33]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.linalg import Vectors
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import Normalizer
from pyspark.ml import Pipeline

vectorAssembler = VectorAssembler(inputCols=[
                                    "HourlyWindDirection",
                                    "ELEVATION",
                                    "HourlyStationPressure"],
                                  outputCol="features")

normalizer = Normalizer(inputCol="features", outputCol="features_norm", p=1.0)

Now we define a function for evaluating our regression prediction performance. We’re using RMSE (Root Mean Squared Error) here , the smaller the better…



In [34]:
def regression_metrics(prediction):
    from pyspark.ml.evaluation import RegressionEvaluator
    evaluator = RegressionEvaluator(
    labelCol="HOURLYWindSpeed", predictionCol="prediction", metricName="rmse")
    rmse = evaluator.evaluate(prediction)
    print("RMSE on test data = %g" % rmse)

Let’s run a linear regression model first for building a baseline.



In [47]:
from pyspark.ml.regression import LinearRegression


lr = LinearRegression(labelCol="HOURLYWindSpeed", featuresCol='features', maxIter=100, regParam=0.0, elasticNetParam=0.0)
pipeline = Pipeline(stages=[vectorAssembler_clean, normalizer_clean, lr])
model = pipeline.fit(df_train_clean)
prediction = model.transform(df_test_clean)
regression_metrics(prediction)

RMSE on test data = 2.68351


In [37]:
#LR1

from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import Normalizer
from pyspark.ml import Pipeline

# 1. Split from df_filtered_safe (the clean dataset) to avoid cast errors
splits_clean = df_filtered_safe.randomSplit([0.8, 0.2], seed=42)
df_train_clean = splits_clean[0]
df_test_clean = splits_clean[1]

# 2. Correct vectorAssembler to use the clean uppercase columns and ELEVATION
vectorAssembler_clean = VectorAssembler(inputCols=[
                                    "HOURLYWindDirection",
                                    "ELEVATION",
                                    "HOURLYStationPressure"],
                                  outputCol="features")

normalizer_clean = Normalizer(inputCol="features", outputCol="features_norm", p=1.0)

# 3. Fit and evaluate the model using the corrected pipeline stages and clean data
lr = LinearRegression(labelCol="HOURLYWindSpeed", featuresCol='features_norm', maxIter=100, regParam=0.0, elasticNetParam=0.0)
pipeline = Pipeline(stages=[vectorAssembler_clean, normalizer_clean, lr])
model = pipeline.fit(df_train_clean)
prediction = model.transform(df_test_clean)
regression_metrics(prediction)

RMSE on test data = 2.68306


Now we’ll try a Gradient Boosted Tree Regressor


In [39]:
#GBT1

from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline

# Use the safe, sanitized clean data splits
# And reference the corrected vectorAssembler_clean and normalizer_clean stages
gbt = GBTRegressor(labelCol="HOURLYWindSpeed", maxIter=100)
pipeline = Pipeline(stages=[vectorAssembler_clean, normalizer_clean, gbt])

model = pipeline.fit(df_train_clean)
prediction = model.transform(df_test_clean)
regression_metrics(prediction)

RMSE on test data = 2.67141


Now let’s switch gears. Previously, we tried to predict HOURLYWindSpeed, but now we predict HOURLYWindDirection. In order to turn this into a classification problem we discretize the value using the Bucketizer. The new feature is called HOURLYWindDirectionBucketized.


In [40]:
from pyspark.ml.feature import Bucketizer, OneHotEncoder
bucketizer = Bucketizer(splits=[ 0, 180, float('Inf') ],inputCol="HOURLYWindDirection", outputCol="HOURLYWindDirectionBucketized")
encoder = OneHotEncoder(inputCol="HOURLYWindDirectionBucketized", outputCol="HOURLYWindDirectionOHE")


Again, we define a function in order to assess how we perform. Here we just use the accuracy measure which gives us the fraction of correctly classified examples. Again, 0 is bad, 1 is good.


In [41]:
def classification_metrics(prediction):
    from pyspark.ml.evaluation import MulticlassClassificationEvaluator
    mcEval = MulticlassClassificationEvaluator().setMetricName("accuracy") .setPredictionCol("prediction").setLabelCol("HOURLYWindDirectionBucketized")
    accuracy = mcEval.evaluate(prediction)
    print("Accuracy on test data = %g" % accuracy)

Again, for baselining we use LogisticRegression.


In [43]:
#LGReg1

from pyspark.ml.classification import LogisticRegression
lr = LogisticRegression(labelCol="HOURLYWindDirectionBucketized", maxIter=10)
#,"ELEVATION","HOURLYStationPressure","HOURLYPressureTendency","HOURLYPrecip"

vectorAssembler = VectorAssembler(inputCols=["HOURLYWindSpeed","HOURLYDRYBULBTEMPC"],
                                  outputCol="features")

# Use the safe clean splits and corrected normalizer
pipeline = Pipeline(stages=[bucketizer,vectorAssembler,normalizer_clean,lr])
model = pipeline.fit(df_train_clean)
prediction = model.transform(df_test_clean)
classification_metrics(prediction)

Accuracy on test data = 0.668675


Let’s try some other Algorithms and see if model performance increases. It’s also important to tweak other parameters like parameters of individual algorithms (e.g. number of trees for RandomForest) or parameters in the feature engineering pipeline, e.g. train/test split ratio, normalization, bucketing, …


In [46]:
#RF1

from pyspark.ml.classification import RandomForestClassifier
rf = RandomForestClassifier(labelCol="HOURLYWindDirectionBucketized", numTrees=100)

vectorAssembler = VectorAssembler(inputCols=["HOURLYWindSpeed","HOURLYDRYBULBTEMPC","ELEVATION","HOURLYStationPressure","HOURLYPressureTendency","HOURLYPrecip"],
                                  outputCol="features")

pipeline = Pipeline(stages=[bucketizer,vectorAssembler,normalizer_clean,rf])
model = pipeline.fit(df_train_clean)
prediction = model.transform(df_test_clean)
classification_metrics(prediction)

Accuracy on test data = 0.696787


In [45]:
#GBT2

from pyspark.ml.classification import GBTClassifier
gbt = GBTClassifier(labelCol="HOURLYWindDirectionBucketized", maxIter=100)

vectorAssembler = VectorAssembler(inputCols=["HOURLYWindSpeed","HOURLYDRYBULBTEMPC","ELEVATION","HOURLYStationPressure","HOURLYPressureTendency","HOURLYPrecip"],
                                  outputCol="features")

pipeline = Pipeline(stages=[bucketizer,vectorAssembler,normalizer_clean,gbt])
model = pipeline.fit(df_train_clean)
prediction = model.transform(df_test_clean)
classification_metrics(prediction)

Accuracy on test data = 0.692771


### Thank you for completing this lab!

This notebook was created by <a href="https://linkedin.com/in/romeo-kienzler-089b4557?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkML0201ENSkillsNetwork953-2023-01-01"> Romeo Kienzler </a>  I hope you found this lab interesting and educational. Feel free to contact me if you have any questions!



## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2020-09-29  | 2.0  | Srishti  |  Migrated Lab to Markdown and added to course repo in GitLab |



<hr>

## <h3 align="center"> © IBM Corporation 2020. All rights reserved. <h3/>
